## Running Ironhack notebooks in VS Code

### One-off setup (already done)

Your `.venv_airflow` environment already has `ipykernel` and `pandas` installed, so you don't need to repeat this. It's here in case you ever make a new environment or need a new package.

1. In the VS Code terminal, make sure you're in `~/data_course`.
2. Switch the environment on: `source .venv_airflow/bin/activate` (the prompt should start with `(.venv_airflow)`).
3. Install what notebooks need: `pip install ipykernel pandas`
4. For any other package a lab needs later (for example numpy), use the same pattern: `pip install <package name>`

### Every time you add a new notebook

Each new .ipynb file needs its kernel picking once. After that, VS Code remembers it for that file.

1. **Drag it in.** Drag the downloaded .ipynb from Windows into your `01_notebooks` folder in the VS Code Explorer (left-hand panel). Rename it if you like (right-click → Rename), for example replacing DATE with the date.
2. **Open it.** Click the file in the Explorer.
3. **Click Select Kernel.** It's in the top-right corner of the notebook. (If it already says `.venv_airflow`, skip to step 7.)
4. **Click Python Environments…**
5. **Pick .venv_airflow.** Choose the line ending `.venv_airflow/bin/python`. Ignore the one marked "Recommended" (`/usr/bin/python3`): that's Ubuntu's own Python and it won't work.
6. **Check the top-right.** It should now say `.venv_airflow`.
7. **Run the import cell.** Click ▷ next to the import cell. A green tick and [1] mean it's working.

## Part 1 — Airflow and dbt Responsibilities

### Q. 1 — Choose the Right Tool

For each requirement, choose Airflow, dbt, or both.

```
Requirement	Tool
Schedule a daily workflow	
Build a SQL transformation model	
Run a dbt test after a dbt model	
Show task failure and retry state	
Define model lineage with ref()	
```

========================================================================
ANSWER Q1
========================================================================

Requirement: Schedule a daily workflow

Tool: Airflow.

====================================

Requirement: Build a SQL transformation model

Tool: dbt.

====================================

Requirement: Run a dbt test after a dbt model

Tool: Both.

====================================

Requirement: Show task failure and retry state

Tool: Airflow.

====================================

Requirement: Define model lineage with ref()

Tool: dbt.

========================================================================

### Q. 2 — Build an Airflow Task that Runs dbt

Create a complete Airflow DAG with one task:

```python
run_dbt
```

The task should run:

```python
dbt run --select fct_daily_sales
```

Use BashOperator.

In [ ]:
# Note: I'm using the work I already did on 26.11.14 to make all of the questions run without having to build eveyrthing from scratch. 
# So this '26.11.17_LAB_Airflow_and_dbt_End-to-End_Pipeline_Integration' folder has a copy of '26.11.14_britmart_dbt''s work. 

# I needed the "raw" schema in this project's DuckDB, so I copied over dev.duckdb from 26.11.14.
# To test the bash_command I first ran "dbt run --select +fct_daily_sales". The "+" in front
# builds the model plus everything upstream it depends on, so the plain Q2 command then works.

from airflow.sdk import DAG
from airflow.sdk import BaseHook, Variable

from airflow.providers.standard.operators.bash import BashOperator
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.python import BranchPythonOperator
from airflow.providers.standard.sensors.filesystem import FileSensor 

from datetime import datetime, timedelta
from pathlib import Path
# Imported all the libaraies I need for this lab, not sure thats best practise.

DBT_PROJECT = Path(__file__).parent.parent
# Works out the dbt project folder from where this DAG file lives. __file__ is this
# file, the first .parent is the 02_dags folder and the second .parent is the project
# folder. This means the command still works if the project folder is renamed or moved.
# The below python will allow this to work no matter how deep the file was...
# DBT_PROJECT = next(folder for folder in Path(__file__).parents if (folder / "dbt_project.yml").exists())


with DAG(
    dag_id = "26.11.17_q02",
    start_date = datetime(2026, 11, 16),
    schedule = None,
    catchup = False,
    #default_args = default_args
    #max_active_runs = 1,
    tags = ["26.11.17"]
) as dag:

    # run_dbt = BashOperator(task_id = "run_dbt", bash_command = f"cd {DBT_PROJECT} && dbt run --select fct_daily_sales")
    # The bash_command is the terminal command Airflow runs for this task, exactly as if
    # I typed it myself. "cd" moves into my dbt project folder, because dbt looks for
    # dbt_project.yml in the folder it runs from and Airflow otherwise starts in a random
    # temporary folder. "&&" means only run the next command if the cd worked, so dbt
    # never runs in the wrong place. "dbt run --select fct_daily_sales" then builds just
    # the fct_daily_sales model in DuckDB rather than the whole project. If dbt succeeds
    # the task goes green; if it fails the task goes red and the dbt error shows in the
    # task log. Airflow decides when the command runs and whether it worked, while dbt
    # does the actual transformation.

    # The correct final command to skip the above terminal faff should have been the below. The above exists beacuse of the wording of the lab question. 
    run_dbt = BashOperator(task_id = "run_dbt", bash_command = f"cd {DBT_PROJECT} && dbt run --select +fct_daily_sales")


### Q. 3 — Create Separate Run and Test Tasks

Create a complete DAG with:

```python
dbt_run
dbt_test
```

Commands:

```python
dbt run --select marts
dbt test --select marts
```

Dependency:

```python
dbt_run -> dbt_test
```

In [ ]:
# Note: I'm using the work I already did on 26.11.14 to make all of the questions run without having to build eveyrthing from scratch. 
# So this '26.11.17_LAB_Airflow_and_dbt_End-to-End_Pipeline_Integration' folder has a copy of '26.11.14_britmart_dbt''s work. 

from airflow.sdk import DAG
from airflow.sdk import BaseHook, Variable

from airflow.providers.standard.operators.bash import BashOperator
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.python import BranchPythonOperator
from airflow.providers.standard.sensors.filesystem import FileSensor 

from datetime import datetime, timedelta
from pathlib import Path
# Imported all the libaraies I need for this lab, not sure thats best practise.

DBT_PROJECT = Path(__file__).parent.parent
# Works out the dbt project folder from where this DAG file lives. __file__ is this
# file, the first .parent is the 02_dags folder and the second .parent is the project
# folder. This means the command still works if the project folder is renamed or moved.
# The below python will allow this to work no matter how deep the file was...
# DBT_PROJECT = next(folder for folder in Path(__file__).parents if (folder / "dbt_project.yml").exists())

with DAG(
    dag_id = "26.11.17_q03",
    start_date = datetime(2026, 11, 16),
    schedule = None,
    catchup = False,
    #default_args = default_args
    #max_active_runs = 1,
    tags = ["26.11.17"]
) as dag:

    dbt_run = BashOperator(task_id = "dbt_run", bash_command = f"cd {DBT_PROJECT} && dbt run --select marts")

    dbt_test = BashOperator(task_id = "dbt_test", bash_command = f"cd {DBT_PROJECT} && dbt test --select marts")

    dbt_run >> dbt_test

# I tested both commands in the terminal first to check dbt works before triggering the DAG in Airflow.
# No "+" needed because the upstream models (staging and q9) were already built when I ran
# "+fct_daily_sales" for Q2, so marts could find everything it depends on.

### Q. 4 — MCQ: Airflow and dbt

Which statement is correct?

A. dbt replaces Airflow scheduling completely

B. Airflow can orchestrate dbt commands while dbt manages transformation logic

C. Airflow defines every SQL model dependency automatically

D. Both tools perform exactly the same job

========================================================================
ANSWER Q4
========================================================================

B. Airflow can orchestrate dbt commands while dbt manages transformation logic.

========================================================================

## Part 2 — CLI Tasks and Controlled Execution

### Q. 5 — Run Only the Required dbt Model

BritMart receives only new order data.

The project contains:

```python
stg_orders
stg_customers
fct_daily_sales
dim_customer
```

Only the order-related sales flow needs to run.

Write a dbt command that selects:

```python
fct_daily_sales
```

and its upstream dependencies.

========================================================================
ANSWER Q5
========================================================================

select_fct_daily_sales = BashOperator(task_id = "select_fct_daily_sales", bash_command = (f"cd {DBT_PROJECT} && dbt run --select +fct_daily_sales"))

'The dbt command is 'dbt run --select +fct_daily_sales'.'

========================================================================

### Q. 6 — Build an Independent dbt CLI Task

Create a complete DAG that runs:

```python
dbt build --select +fct_daily_sales
```

Use one Airflow task named:

```python
build_sales_models
```

In [ ]:
# Note: I'm using the work I already did on 26.11.14 to make all of the questions run without having to build eveyrthing from scratch. 
# So this '26.11.17_LAB_Airflow_and_dbt_End-to-End_Pipeline_Integration' folder has a copy of '26.11.14_britmart_dbt''s work. 

from airflow.sdk import DAG
from airflow.sdk import BaseHook, Variable

from airflow.providers.standard.operators.bash import BashOperator
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.python import BranchPythonOperator
from airflow.providers.standard.sensors.filesystem import FileSensor 

from datetime import datetime, timedelta
from pathlib import Path
# Imported all the libaraies I need for this lab, not sure thats best practise.

DBT_PROJECT = Path(__file__).parent.parent
# Works out the dbt project folder from where this DAG file lives. __file__ is this
# file, the first .parent is the 02_dags folder and the second .parent is the project
# folder. This means the command still works if the project folder is renamed or moved.
# The below python will allow this to work no matter how deep the file was...
# DBT_PROJECT = next(folder for folder in Path(__file__).parents if (folder / "dbt_project.yml").exists())

with DAG(
    dag_id = "26.11.17_q06",
    start_date = datetime(2026, 11, 16),
    schedule = None,
    catchup = False,
    #default_args = default_args
    #max_active_runs = 1,
    tags = ["26.11.17"]
) as dag:

    dbt_build = BashOperator(task_id = "build_sales_models", bash_command = f"cd {DBT_PROJECT} && dbt build --select +fct_daily_sales")

# I tested both commands in the terminal first to check dbt works before triggering the DAG in Airflow.
# dbt build --select +fct_daily_sales


### Q. 7 — Decide When Containerised Execution Helps

For each statement, mark useful or not the main reason for containerised execution.

```python
A. Keep the same dbt version and dependencies across environments.
B. Avoid documenting which image version is used.
C. Package dbt and its dependencies together.
D. Remove all need for credentials and network access.
```

# Containerised execution means running dbt inside a container (usually Docker) rather than directly on my machine. A container is a sealed, portable box built from an image that holds the exact dbt version, the database adapter, Python and any other libraries it needs, so it runs the same way on my laptop, a colleague's machine or a server. Airflow can start the container, run a dbt command inside it, then throw it away, using something like DockerOperator instead of BashOperator. The main benefit is consistency, since everyone runs exactly the same versions. A container doesn't remove the need for credentials or network access, though: it still needs a password and a connection to reach the database.


========================================================================
ANSWER Q7
========================================================================

A. Keep the same dbt version and dependencies across environments.

Useful.

====================================

B. Avoid documenting which image version is used.

Not the main reason.

====================================

C. Package dbt and its dependencies together.

Useful.

====================================

D. Remove all need for credentials and network access.

Not the main reason.

========================================================================

### Q. 8 — MCQ: Separate Run and Test Tasks

Why can separate dbt run and dbt test Airflow tasks be useful?

A. They make every data test pass automatically

B. They remove the need for dbt models

C. They make it easier to identify whether model execution or testing failed

D. They prevent Airflow from writing logs

========================================================================
ANSWER Q8
========================================================================

C. They make it easier to identify whether model execution or testing failed.

========================================================================

## Part 3 — Dependencies, Data Readiness, and Publication Gates

### Q. 9 — Design the Dependency Chain

BritMart has:

```python
load_source
dbt_run
dbt_test
publish_report
```

Write the correct dependency chain.

Then explain why `publish_report` should wait for `dbt_test`.

========================================================================
ANSWER Q9
========================================================================


load_source >> dbt_run >> dbt_test >> publish_report


publish_report should wait for dbt_test because the tests act as a publication gate. If the transformed data has a problem, like nulls, duplicates or unexpected values, the test fails and the report never gets published. That stops bad data from reaching the people who use the report.

========================================================================

### Q. 10 — Build a Publication-Gate DAG

Create a complete DAG:

```python
load_source
run_dbt
test_dbt
publish
```

Use BashOperator for all tasks.

Commands:

```python
load_source -> echo source ready
run_dbt -> dbt run --select marts
test_dbt -> dbt test --select marts
publish -> echo publish report
```

In [ ]:
# Note: I'm using the work I already did on 26.11.14 to make all of the questions run without having to build eveyrthing from scratch. 
# So this '26.11.17_LAB_Airflow_and_dbt_End-to-End_Pipeline_Integration' folder has a copy of '26.11.14_britmart_dbt''s work. 

from airflow.sdk import DAG
from airflow.sdk import BaseHook, Variable

from airflow.providers.standard.operators.bash import BashOperator
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.python import BranchPythonOperator
from airflow.providers.standard.sensors.filesystem import FileSensor 

from datetime import datetime, timedelta
from pathlib import Path
# Imported all the libaraies I need for this lab, not sure thats best practise.

DBT_PROJECT = Path(__file__).parent.parent
# Works out the dbt project folder from where this DAG file lives. __file__ is this
# file, the first .parent is the 02_dags folder and the second .parent is the project
# folder. This means the command still works if the project folder is renamed or moved.
# The below python will allow this to work no matter how deep the file was...
# DBT_PROJECT = next(folder for folder in Path(__file__).parents if (folder / "dbt_project.yml").exists())

with DAG(
    dag_id = "26.11.17_q10",
    start_date = datetime(2026, 11, 16),
    schedule = None,
    catchup = False,
    #default_args = default_args
    #max_active_runs = 1,
    tags = ["26.11.17"]
) as dag:

    load_source = BashOperator(task_id = "load_source", bash_command = ('echo "Source ready."'))

    run_dbt = BashOperator(task_id = "run_dbt", bash_command = f"cd {DBT_PROJECT} && dbt run --select marts")

    test_dbt = BashOperator(task_id = "test_dbt", bash_command = f"cd {DBT_PROJECT} && dbt test --select marts")

    publish = BashOperator(task_id = "publish", bash_command = ('echo "Publish report."'))

    load_source >> run_dbt >> test_dbt >> publish

# I used Airflow's built-in "dags test" command to run the whole DAG once from the terminal, not just the dbt commands.
# It should end with "DagRun Finished: dag_id=26.11.17_q10 ... state=success".
# cd ~/data_course
# source .venv_airflow/bin/activate
# airflow dags test 26.11.17_q10
# I can also check the file has no errors with: python ~/data_course/26.11.17_LAB_Airflow_and_dbt*/02_dags/26.11.17_q10.py
# If nothing is returned, the file is fine.

### Q. 11 — Choose the Correct dbt Selection

BritMart has two business areas:

```python
sales
inventory
```

A new sales file arrives. No inventory data changed.

Which approach is better?

```python
A. Run every dbt model in the project.
B. Select only the sales models and required dependencies.
```

Explain your choice.

========================================================================
ANSWER Q11
========================================================================

(I actually asked this myself when doing the DAG for Q3. My first thought was why not just put + in front of every dbt command to make sure everything upstream was run every single time. I guess you have to be aware of the lineage of your tables when you run dbt.)

B: Select only the sales models and their required dependencies.

Only sales data changed, so rebuilding the inventory models would waste time and compute without changing anything. Selecting just the sales models (for example with a tag or `+model` selector) makes the run faster and cheaper. It also means an unrelated inventory problem can't block the sales report.

========================================================================

### Q. 12 — MCQ: Publication Gate

What should happen if `dbt_test` fails?

A. Stop downstream publication until the failure is understood

B. Skip the test result and mark the workflow successful

C. Publish the report anyway

D. Delete all dbt models

========================================================================
ANSWER Q12
========================================================================

A. Stop downstream publication until the failure is understood.

(For question 12 the answer is A... but doesn't the downstream publication stop as soon as a failure happens anyway? Yes, if the DAG is set up properly. By default, every Airflow task has the trigger rule all_success: it only runs if everything upstream succeeded. So when dbt_test fails, publish is automatically marked upstream_failed and never runs. That's exactly what I built in Q10.)

========================================================================

## Part 4 — Retries, Failure Classification, Logging, and Recovery

### Q. 13 — Retry or Fix First?

Choose whether an automatic retry is useful.

```
Failure	Retry useful?
Temporary database timeout	
Invalid SQL in a dbt model	
Short network interruption	
dbt test fails because customer_id contains nulls	
```

========================================================================
ANSWER Q13
========================================================================

Failure: Temporary database timeout

Retry useful?: Yes.

====================================

Failure: Invalid SQL in a dbt model

Retry useful?: No.

====================================

Failure: Short network interruption

Retry useful?: Yes.

====================================

Failure: dbt test fails because customer_id contains nulls

Retry useful?: No.

========================================================================

### Q. 14 — Add a Retry to a dbt Task

Create a complete Airflow DAG with one task:

```python
run_dbt
```

Requirements:

```python
retries = 2
retry_delay = 2 minutes
command = dbt run --select fct_daily_sales
```

In [ ]:
# Note: I'm using the work I already did on 26.11.14 to make all of the questions run without having to build eveyrthing from scratch. 
# So this '26.11.17_LAB_Airflow_and_dbt_End-to-End_Pipeline_Integration' folder has a copy of '26.11.14_britmart_dbt''s work. 

from airflow.sdk import DAG
from airflow.sdk import BaseHook, Variable

from airflow.providers.standard.operators.bash import BashOperator
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.python import BranchPythonOperator
from airflow.providers.standard.sensors.filesystem import FileSensor 

from datetime import datetime, timedelta
from pathlib import Path
# Imported all the libaraies I need for this lab, not sure thats best practise.

DBT_PROJECT = Path(__file__).parent.parent
# Works out the dbt project folder from where this DAG file lives. __file__ is this
# file, the first .parent is the 02_dags folder and the second .parent is the project
# folder. This means the command still works if the project folder is renamed or moved.
# The below python will allow this to work no matter how deep the file was...
# DBT_PROJECT = next(folder for folder in Path(__file__).parents if (folder / "dbt_project.yml").exists())

with DAG(
    dag_id = "26.11.17_q14",
    start_date = datetime(2026, 11, 16),
    schedule = None,
    catchup = False,
    #default_args = default_args
    #max_active_runs = 1,
    tags = ["26.11.17"]
) as dag:

    run_dbt = BashOperator(task_id = "run_dbt", bash_command = (f"cd {DBT_PROJECT} && dbt run --select fct_daily_sales"), retries=2, retry_delay=timedelta(minutes=2))

# airflow dags test 26.11.17_q14
# It should end with "DagRun Finished: dag_id=26.11.17_q14 ... state=success".

### Q. 15 — Read a dbt Failure Log

A task log contains:

```python
Running with dbt...
Found 12 models
Database Error in model fct_daily_sales
column "amount_gbp" does not exist
Task exited with return code 1
```

Which line gives the best starting point for debugging?

What would you check?

========================================================================
ANSWER Q15
========================================================================

I don't like Ironhack's answer, although it's basically the same thing I'm thinking. I'd start with 'Database Error in model fct_daily_sales', as you need to know where the issue is before you can look at the specifics. Ironhack says 'column "amount_gbp" does not exist'.

========================================================================

### Q. 16 — MCQ: Retry

Which failure is the best candidate for an automatic retry?

A. A missing required model file

B. A dbt model with invalid SQL syntax

C. A failed uniqueness test caused by duplicate business data

D. A temporary network timeout

========================================================================
ANSWER Q16
========================================================================

D. A temporary network timeout.

========================================================================

## Optional Practice

### Q. 17 — Build a Complete Airflow + dbt Workflow

Create a complete DAG with:

```python
check_source
run_sales_models
test_sales_models
publish_sales
```

Requirements:

```python
schedule = @daily
catchup = False
run_sales_models retries = 1
test must pass before publish
```

Commands:

```python
echo source ready
dbt run --select +fct_daily_sales
dbt test --select +fct_daily_sales
echo sales published
```

In [ ]:
# Note: I'm using the work I already did on 26.11.14 to make all of the questions run without having to build eveyrthing from scratch. 
# So this '26.11.17_LAB_Airflow_and_dbt_End-to-End_Pipeline_Integration' folder has a copy of '26.11.14_britmart_dbt''s work. 

from airflow.sdk import DAG
from airflow.sdk import BaseHook, Variable

from airflow.providers.standard.operators.bash import BashOperator
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.python import BranchPythonOperator
from airflow.providers.standard.sensors.filesystem import FileSensor 

from datetime import datetime, timedelta
from pathlib import Path
# Imported all the libaraies I need for this lab, not sure thats best practise.

DBT_PROJECT = Path(__file__).parent.parent
# Works out the dbt project folder from where this DAG file lives. __file__ is this
# file, the first .parent is the 02_dags folder and the second .parent is the project
# folder. This means the command still works if the project folder is renamed or moved.
# The below python will allow this to work no matter how deep the file was...
# DBT_PROJECT = next(folder for folder in Path(__file__).parents if (folder / "dbt_project.yml").exists())

with DAG(
    dag_id = "26.11.17_q17",
    start_date = datetime(2026, 11, 16),
    schedule = "@daily",
    catchup = False,
    #default_args = default_args
    #max_active_runs = 1,
    tags = ["26.11.17"]
) as dag:

    check_source = BashOperator(task_id = "check_source", bash_command = ('echo "Source ready."'))

    run_sales_models = BashOperator(task_id = "run_sales_models", bash_command = (f"cd {DBT_PROJECT} && dbt run --select +fct_daily_sales"), retries=1, retry_delay=timedelta(minutes=2))

    test_sales_models = BashOperator(task_id = "test_sales_models", bash_command = (f"cd {DBT_PROJECT} && dbt test --select +fct_daily_sales"))

    publish_sales = BashOperator(task_id = "publish_sales", bash_command = ('echo "Sales published."'))

    check_source >> run_sales_models >> test_sales_models >> publish_sales


### Q. 18 — Diagnose the Wrong Task Order

A DAG uses:

```python
publish_report
    ↓
dbt_test
    ↓
dbt_run
```

Explain the problem and provide the correct order.

========================================================================
ANSWER Q18
========================================================================

The order is reversed, the problem is self-explanatory.

========================================================================

### Q. 19 — Decide What Evidence to Keep

A dbt task fails in Airflow.

Choose at least four useful pieces of evidence to retain:

```python
Airflow task log
dbt command
selected models
dbt error message
DAG run time
business output count
random screenshot unrelated to the failure
```

========================================================================
ANSWER Q19
========================================================================

Airflow task log
dbt command
dbt error message
business output count

But anything other than 'random screenshot unrelated to the failure' is ok. 

========================================================================

### Q. 20 — Design a Recovery Flow

A daily workflow shows:

```python
load_source = success
dbt_run = success
dbt_test = failed
publish_report = upstream_failed
```

Write the recovery steps.

========================================================================
ANSWER Q20
========================================================================

'A sensible recovery flow is:

- Open the dbt_test task log.

- Identify which test failed.

- Check whether the failure is caused by bad data or an incorrect test.

- Correct the source, transformation, or test definition.

- Rerun the required dbt work.

- Rerun the test.

- Allow publication only after the test passes.

- Check the final published output.'

========================================================================